# 🥉 03_ingestao_bronze — Camada Bronze de `ecommerce_pedidos`
**Squad 2 · Dupla 3** · Passo 5 do plano · regras 1 e 2 da planilha (e a conferência de **tipo** das regras 3 e 4)

Copia para a Bronze cada micro-lote **novo** do `raw`, como chegou, com as colunas de auditoria do Passo 5. Executa **uma rodada** por chamada:
pode ser rodado sozinho ou repetidamente pelo `06_orquestrador`.

| Regra da planilha | Como a Bronze aplica |
|---|---|
| 1 · as 6 colunas obrigatórias existem em todo micro-lote | arquivo sem alguma delas não entra e fica anotado com o motivo |
| 2 · não processar o mesmo arquivo duas vezes | o caminho completo do arquivo (`bronze_source_file`) é o controle de estado |
| 3 · `valor_total` do tipo float/decimal | coluna que chega como texto recusa o arquivo; o "> 0" é conferido na Silver, pedido a pedido |
| 4 · `dt_pedido` lido como data e hora (ISO 8601) | data já tipada entra direto; data em texto entra se **todo** valor for ISO 8601 válido |

| | |
|---|---|
| Entrada | `raw/real-time-data/AAAA/MM/DD/HHMMSS/ecommerce_pedidos.parquet` (somente leitura) |
| Saída | `squad2/bronze/ecommerce_pedidos` (Delta no Azure, `append`, partição ano/mês/dia/hora da ingestão) |
| Controle (regra 2) | a própria Bronze (`bronze_source_file`) + recusados em `metadata/ecommerce_pedidos/bronze_ignorados.json` |

| Etapa | Conteúdo |
|---|---|
| 1 | Configuração e contrato da EDA |
| 2 | Descoberta dos arquivos novos (regra 2) |
| 3 | Leitura e validação por arquivo (regras 1, 3 e 4) |
| 4 | Etiquetagem: auditoria e partições |
| 5 | Gravação no Azure |
| 6 | Conferência |
| 7 | Resumo para o orquestrador |

Cada etapa **faz** o trabalho e **valida** em seguida: ✅ ok · ⚠️ achado (segue) · ❌ erro (para).

## Etapa 1 — Configuração
**Faz:** carrega o `00_setup_config` em modo rápido, o contrato publicado pela EDA (colunas e tipos esperados) e os parâmetros da ingestão.
**Valida:** se as funções necessárias existem; sem contrato, segue conferindo só as colunas obrigatórias da regra 1.

In [0]:
VALIDACAO_COMPLETA = False

In [0]:
%run ./00_setup_config

# ⚙️ 00_setup_config — Configuração central
**Squad 2 · Dupla 3 · `ecommerce_pedidos`**

Prepara endereços, credenciais, conexões e funções usadas por todo o pipeline. **Não lê nem grava pedidos.**

Cada etapa **faz o trabalho e valida em seguida**: ✅ ok · ⚠️ aviso (segue) · ❌ erro (para e explica).

**Uso nos outros notebooks** (duas células; o `%run` fica sozinho):
```
VALIDACAO_COMPLETA = False
```
```
%run ./00_setup_config
```
Dependências em **Environment → Dependencies**: `azure-identity`, `azure-storage-file-datalake`.

## Etapa 1 — Ambiente e endereços
**Faz:** cria a função `checar` e guarda os nomes fixos do projeto.
**Valida:** se está no Databricks e se origem e destino são containers diferentes.

Modo de validação: RÁPIDA
✅ Rodando no Databricks
✅ Origem (raw) e destino (squad2) separados


## Etapa 2 — Credenciais
**Faz:** busca as 7 chaves no Secret Scope. Ficam só na memória (aparecem como `[REDACTED]` se impressas).
**Valida:** se o cofre existe, se nenhuma chave falta e se o host do SQL tem o formato certo.

✅ Cofre 'mercadata-ecommerce-pedidos' encontrado
✅ 7 credenciais carregadas (valores ocultos)
✅ SQL_HOST no formato do Azure SQL


...

✅ url_origem OK
✅ tabela_delta_existe OK (Bronze já existe)
✅ ler_metadata OK (None para anotação inexistente)


## Etapa 6 — Funções do SQL Server
**Faz:** cria as funções de gravar e consultar o banco. Gravação pelo conector `sqlserver` (o `jdbc` é bloqueado no Serverless); leitura pelo `jdbc`.
**Valida:** se toda tabela recebe o schema `squad2`.

✅ Tabelas SQL vão para o schema 'squad2'


## Etapa 7 — Teste completo (só ao rodar este notebook direto)
**Faz e valida:** varre o `raw` · lê o micro-lote mais recente e confere colunas e tipos (regra 1) · grava e relê em `metadata/` · `SELECT 1` no SQL Server · confere se o schema `squad2` existe no banco.
No modo rápido (`%run`), é pulada. No fim, limpa as variáveis temporárias.

Teste completo pulado (modo rápido).
⚙️ 00_setup_config pronto para uso


In [0]:
from pyspark.sql import functions as F

SCHEMA_ESPERADO = (ler_metadata("eda_raw") or {}).get("schema", {})   # {coluna: tipo}; vazio = sem contrato da EDA
LIMITE_ARQUIVOS = 500                    # teto de micro-lotes por rodada: limita a memória quando há muito acumulado
ANOTACAO_IGNORADOS = "bronze_ignorados"  # metadata/ecommerce_pedidos/bronze_ignorados.json
PARTICOES = ["ano", "mes", "dia", "hora"]

# Tipos do pandas que cabem sem perda no tipo do contrato (dtype.kind: i inteiro, u inteiro sem sinal, f decimal, M data, O texto, b lógico)
COMPATIVEIS = {"double": "iuf", "bigint": "iu", "timestamp": "M", "string": "O", "boolean": "b"}
REGRA_DA_COLUNA = {"valor_total": "regra 3", "dt_pedido": "regra 4"}   # tipo errado nessas colunas cita a regra da planilha

# Resumo devolvido ao orquestrador na Etapa 7; cada etapa preenche a sua parte
resumo = {"arquivos_novos": 0, "arquivos_gravados": 0, "linhas_gravadas": 0, "rejeitados": 0, "falhas_leitura": 0, "arquivos": []}

# ------------------------------ validação ------------------------------
checar(all(n in globals() for n in ["listar_arquivos_origem", "ler_delta", "escrever_delta", "ler_metadata", "escrever_metadata"]),
       "Funções do 00_setup_config disponíveis", "Rode o %run ./00_setup_config antes desta célula")
checar(set(COLUNAS_OBRIGATORIAS) <= set(SCHEMA_ESPERADO) if SCHEMA_ESPERADO else True,
       f"Regra 1: {len(COLUNAS_OBRIGATORIAS)} colunas obrigatórias ({', '.join(COLUNAS_OBRIGATORIAS)})", "Contrato da EDA sem alguma coluna obrigatória")
checar(bool(SCHEMA_ESPERADO), f"Contrato da EDA carregado ({len(SCHEMA_ESPERADO)} colunas)",
       "Contrato da EDA não encontrado: conferindo só as colunas obrigatórias", parar=False)
print(f"Destino: {CAMINHO_BRONZE}")

✅ Funções do 00_setup_config disponíveis
✅ Regra 1: 6 colunas obrigatórias (id_pedido, id_cliente, dt_pedido, status_pedido, valor_total, metodo_pagamento)
✅ Contrato da EDA carregado (10 colunas)
Destino: abfss://squad2@internshipdatalake.dfs.core.windows.net/bronze/ecommerce_pedidos


## Etapa 2 — Descoberta dos arquivos novos
**Faz:** novos = arquivos no `raw` − arquivos já gravados na Bronze − arquivos já rejeitados. O controle está nos próprios dados: se um arquivo está na Bronze, ele entrou por completo (o Delta só registra gravações concluídas).
**Valida:** nenhum arquivo "novo" já existe na Bronze.

In [0]:
no_raw = listar_arquivos_origem()
bronze_existe = tabela_delta_existe(CAMINHO_BRONZE)

# Caminhos já ingeridos: uma linha por arquivo, não por pedido (cabe na memória mesmo com o histórico crescendo)
ja_gravados = ({r["bronze_source_file"] for r in ler_delta(CAMINHO_BRONZE).select("bronze_source_file").distinct().collect()}
               if bronze_existe else set())
ignorados = ler_metadata(ANOTACAO_IGNORADOS) or {}         # {caminho: motivo} de rodadas anteriores
pendentes = [c for c in no_raw if c not in ja_gravados and c not in ignorados]
novos = pendentes[:LIMITE_ARQUIVOS]                         # o restante fica para a próxima rodada
resumo["arquivos_novos"] = len(novos)

# ------------------------------ validação ------------------------------
checar(not set(novos) & ja_gravados, "Nenhum arquivo novo já existe na Bronze (regra 2)", "Arquivo já gravado selecionado como novo")
print(f"raw: {len(no_raw)} · já na Bronze: {len(ja_gravados)} · ignorados: {len(ignorados)} · "
      f"novos nesta rodada: {len(novos)}" + (f" (+{len(pendentes) - len(novos)} na próxima)" if len(pendentes) > len(novos) else ""))
if not bronze_existe:
    print("ℹ️ Bronze ainda não existe: será criada na primeira gravação")

✅ Nenhum arquivo novo já existe na Bronze (regra 2)
raw: 196 · já na Bronze: 196 · ignorados: 0 · novos nesta rodada: 0


## Etapa 3 — Leitura e validação por arquivo (regras 1, 3 e 4)
**Faz:** lê cada arquivo novo **separadamente** e decide se ele pode entrar na Bronze antes de juntá-lo aos outros.
O pandas só abre o arquivo (as datas em nanossegundos não são aceitas pelo Spark); a conversão para PySpark acontece na Etapa 4.

| Situação | Destino |
|---|---|
| Falta uma das 6 colunas obrigatórias (regra 1) | recusado: `bronze_ignorados.json` com o motivo |
| `valor_total` que não é número (regra 3) ou `dt_pedido` que não é data nem texto ISO 8601 válido (regra 4) | recusado |
| Coluna fora do contrato ou outra coluna com tipo diferente do contrato | recusado: o Delta não grava coluna nova nem tipo misturado (limite técnico, não regra de negócio) |
| Falta uma coluna **opcional** (fora das 6 obrigatórias) | aceito: a coluna entra vazia |
| Falha de leitura (rede, arquivo ilegível) | não é anotado: tenta de novo na próxima rodada |

Arquivo recusado continua intacto no `raw`; para reprocessá-lo, basta tirá-lo do `bronze_ignorados.json`.
**Valida:** aceitos + recusados + falhas = arquivos novos.

In [0]:
def validar_arquivo(pdf):
    """(pdf pronto para a Bronze, None) se o micro-lote pode entrar; (None, motivo) se não pode."""
    if pdf.empty:
        return None, "vazio"
    faltam = [c for c in COLUNAS_OBRIGATORIAS if c not in pdf.columns]
    if faltam:
        return None, f"regra 1: faltam as colunas obrigatórias {faltam}"
    if not SCHEMA_ESPERADO:
        return pdf, None
    extras = [c for c in pdf.columns if c not in SCHEMA_ESPERADO]
    if extras:
        return None, f"colunas fora do contrato {extras} (o Delta não grava coluna nova sem alinhamento)"

    for coluna in pdf.columns:
        tipo, valores = SCHEMA_ESPERADO[coluna], pdf[coluna]
        if valores.isna().all() or valores.dtype.kind in COMPATIVEIS.get(tipo, ""):
            continue                                        # coluna vazia ou tipo compatível: entra como chegou
        regra = REGRA_DA_COLUNA.get(coluna, "contrato")
        if tipo == "timestamp" and valores.dtype.kind == "O":
            # Data em texto (regra 4): só entra se TODO valor preenchido for ISO 8601; com fuso explícito, vai para o horário de Brasília dos demais
            try:                                            # format="ISO8601" só existe no pandas 2; no 1.x a leitura padrão já entende ISO 8601
                datas = pd.to_datetime(valores, errors="coerce", **({"format": "ISO8601"} if int(pd.__version__[0]) >= 2 else {}))
            except (ValueError, TypeError):                 # fusos misturados na mesma coluna
                datas = None
            if datas is not None and datas.dtype.kind == "M" and datas[valores.notna()].notna().all():
                pdf[coluna] = datas.dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None) if datas.dt.tz else datas
                continue
            return None, f"{regra}: {coluna} tem valor que não é data ISO 8601"
        return None, f"{regra}: {coluna} chegou como {valores.dtype}, esperado {tipo}"
    return pdf, None


aceitos, novos_ignorados, falhas_leitura, opcionais_ausentes = [], {}, [], {}
for caminho in novos:
    try:
        pdf = pd.read_parquet(io.BytesIO(fs_origem.get_file_client(caminho).download_file().readall()))
    except Exception as erro:                           # não anota: pode ser falha passageira de rede
        falhas_leitura.append(f"{caminho}: {type(erro).__name__}")
        continue
    pdf, motivo = validar_arquivo(pdf)
    if motivo:
        novos_ignorados[caminho] = motivo
        continue
    aceitos.append(pdf.assign(bronze_source_file=caminho))
    ausentes = sorted(set(SCHEMA_ESPERADO) - set(pdf.columns))
    if ausentes:
        opcionais_ausentes[caminho] = ausentes
resumo.update(rejeitados=len(novos_ignorados), falhas_leitura=len(falhas_leitura))

# ------------------------------ validação ------------------------------
checar(len(aceitos) + len(novos_ignorados) + len(falhas_leitura) == len(novos),
       f"{len(aceitos)} aceitos · {len(novos_ignorados)} recusados · {len(falhas_leitura)} falhas de leitura", "Arquivo perdido na validação")
checar(not novos_ignorados, "Nenhum arquivo recusado", f"Recusados: {novos_ignorados}", parar=False)
checar(not opcionais_ausentes, "Todos os aceitos trazem as colunas opcionais",
       f"{len(opcionais_ausentes)} aceitos sem coluna opcional (entram vazias): {opcionais_ausentes}", parar=False)
checar(not falhas_leitura, "Nenhuma falha de leitura", f"Falhas (nova tentativa na próxima rodada): {falhas_leitura}", parar=False)

✅ 0 aceitos · 0 recusados · 0 falhas de leitura
✅ Nenhum arquivo recusado
✅ Todos os aceitos trazem as colunas opcionais
✅ Nenhuma falha de leitura


## Etapa 4 — Etiquetagem
**Faz:** junta os arquivos aceitos e converte para PySpark **uma única vez**, coloca cada coluna no tipo do contrato (coluna opcional ausente entra vazia;
nenhum valor é alterado) e acrescenta só as colunas do Passo 5: `bronze_ingested_at` (momento da ingestão, igual para toda a rodada),
`bronze_source_file` (caminho completo do micro-lote) e `ano`, `mes`, `dia`, `hora` (partição, a partir de `bronze_ingested_at`, em UTC).
**Valida:** nenhuma linha ganha ou perdida e todas as colunas do contrato e de auditoria presentes.

In [0]:
df_bronze, linhas_lote = None, 0
if aceitos:
    pdf_lote = pd.concat(aceitos, ignore_index=True)
    linhas_lote = len(pdf_lote)
    df_lote = spark.createDataFrame(pdf_lote)
    del pdf_lote, aceitos                            # os dados já estão no Spark: libera a cópia pandas
    tipos_lote = dict(df_lote.dtypes)

    def no_contrato(coluna, tipo):
        """Coluna no tipo do contrato. Opcional ausente em todos os arquivos entra vazia; o NaN que o pandas usa para 'vazio' vira nulo."""
        if coluna not in tipos_lote:
            return F.lit(None).cast(tipo).alias(coluna)
        valor = F.when(~F.isnan(coluna), F.col(coluna)) if tipos_lote[coluna] in ("double", "float") else F.col(coluna)
        return valor.cast(tipo).alias(coluna)

    colunas_dado = ([no_contrato(c, t) for c, t in SCHEMA_ESPERADO.items()] if SCHEMA_ESPERADO
                    else [c for c in df_lote.columns if c != "bronze_source_file"])
    df_bronze = (df_lote.select(*colunas_dado, "bronze_source_file")
                 .withColumn("bronze_ingested_at", F.current_timestamp())
                 .withColumn("ano", F.year("bronze_ingested_at"))
                 .withColumn("mes", F.month("bronze_ingested_at"))
                 .withColumn("dia", F.dayofmonth("bronze_ingested_at"))
                 .withColumn("hora", F.hour("bronze_ingested_at")))

# ------------------------------ validação ------------------------------
if df_bronze is None:
    print("Nada para etiquetar nesta rodada.")
else:
    checar(df_bronze.count() == linhas_lote, f"{linhas_lote} linhas etiquetadas (nenhuma ganha ou perdida)", "Linhas alteradas na etiquetagem")
    checar({*SCHEMA_ESPERADO, "bronze_ingested_at", "bronze_source_file", *PARTICOES} <= set(df_bronze.columns),
           "Colunas do contrato, de auditoria e de partição presentes", "Faltam colunas na Bronze")

Nada para etiquetar nesta rodada.


## Etapa 5 — Gravação no Azure
**Faz:** grava os aceitos em `squad2/bronze/ecommerce_pedidos` (Delta, `append`, partição ano/mês/dia/hora) num **único commit**: ou entram todos os arquivos, ou nenhum.
Depois, acrescenta os recusados à anotação em `metadata/`. Sem `mergeSchema`: se o formato mudar, o Delta recusa a gravação em vez de misturar formatos.
**Valida:** a gravação conclui (qualquer erro interrompe a célula); a anotação de recusados é relida.

In [0]:
linhas_antes = ler_delta(CAMINHO_BRONZE).count() if bronze_existe else 0
arquivos_gravados = []
if df_bronze is not None:
    arquivos_gravados = [r["bronze_source_file"] for r in df_bronze.select("bronze_source_file").distinct().collect()]
    escrever_delta(df_bronze, CAMINHO_BRONZE, "append", particionar_por=PARTICOES)
    resumo.update(arquivos_gravados=len(arquivos_gravados), linhas_gravadas=linhas_lote, arquivos=sorted(arquivos_gravados))

# Recusados são anotados depois dos dados: se esta escrita falhar, eles só são reavaliados na próxima rodada
if novos_ignorados:
    escrever_metadata(ANOTACAO_IGNORADOS, {**ignorados, **novos_ignorados})

# ------------------------------ validação ------------------------------
if arquivos_gravados:
    print(f"✅ {len(arquivos_gravados)} arquivos gravados em {CAMINHO_BRONZE}")
if novos_ignorados:
    checar(set(novos_ignorados) <= set(ler_metadata(ANOTACAO_IGNORADOS) or {}), "Recusados anotados em metadata/",
           "Recusados não foram anotados")

## Etapa 6 — Conferência
**Faz:** relê a Bronze depois da gravação.
**Valida:** a Bronze cresceu exatamente o número de linhas gravadas, todos os arquivos da rodada estão lá e **nenhum arquivo aparece com mais de uma ingestão** em toda a tabela (prova da regra 2).

In [0]:
if not arquivos_gravados:
    print("Nada gravado nesta rodada: conferência dispensada.")
else:
    bronze = ler_delta(CAMINHO_BRONZE)
    linhas_depois = bronze.count()
    por_arquivo = bronze.groupBy("bronze_source_file").agg(F.countDistinct("bronze_ingested_at").alias("ingestoes"))
    reingeridos = por_arquivo.filter(F.col("ingestoes") > 1).count()
    presentes = por_arquivo.filter(F.col("bronze_source_file").isin(arquivos_gravados)).count()

    # ------------------------------ validação ------------------------------
    checar(linhas_depois - linhas_antes == linhas_lote, f"Bronze cresceu {linhas_depois - linhas_antes} linhas (= gravadas)",
           f"Bronze cresceu {linhas_depois - linhas_antes}, esperado {linhas_lote}")
    checar(presentes == len(arquivos_gravados), f"Os {presentes} arquivos da rodada estão na Bronze", "Arquivo da rodada ausente na Bronze")
    checar(reingeridos == 0, "Nenhum arquivo com mais de uma ingestão em toda a Bronze", f"{reingeridos} arquivos ingeridos mais de uma vez")

    # ------------------------------ resultado ------------------------------
    display(bronze.filter(F.col("bronze_source_file").isin(arquivos_gravados))
            .groupBy(*PARTICOES, "bronze_ingested_at").agg(F.countDistinct("bronze_source_file").alias("arquivos"), F.count("*").alias("linhas")))
    print(f"Bronze total: {linhas_depois:,} linhas de {por_arquivo.count():,} micro-lotes")

Nada gravado nesta rodada: conferência dispensada.


## Etapa 7 — Resumo para o orquestrador
**Faz:** libera a memória e devolve o resumo da rodada em JSON (`dbutils.notebook.exit`), que o `06_orquestrador` lê a cada ciclo.
Erros de execução já interromperam o notebook nas etapas anteriores; falhas de leitura voltam no resumo e são tentadas de novo.

In [0]:
for _v in ["df_lote", "df_bronze", "bronze", "por_arquivo"]:
    globals().pop(_v, None)
globals().pop("_v", None)

print(json.dumps({k: v for k, v in resumo.items() if k != "arquivos"}, ensure_ascii=False))
dbutils.notebook.exit(json.dumps(resumo, ensure_ascii=False))